# 03 — Monte Carlo reference pricer

We price a European call on an equally weighted five-stock basket. Monte Carlo simulates terminal prices under a correlated geometric Brownian-motion model, calculates the payoff on each path, and averages the discounted payoffs. These prices become the MLP's training targets.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

scenarios = pd.read_csv(Path('data') / 'raw_scenarios.csv')
rng = np.random.default_rng(123)
n_paths = 2_000
n_assets = 5
weights = np.full(n_assets, 1 / n_assets)


In [6]:
def price_one(row, n_paths=2_000):
    spots = row[[f'spot_{i+1}' for i in range(n_assets)]].to_numpy()
    vols = row[[f'vol_{i+1}' for i in range(n_assets)]].to_numpy()
    T, r, K, rho = row['maturity'], row['rate'], row['strike'], row['correlation']

    # Equicorrelation: every pair of assets has correlation rho.
    corr = np.full((n_assets, n_assets), rho)
    np.fill_diagonal(corr, 1.0)
    chol = np.linalg.cholesky(corr)
    independent = rng.standard_normal((n_paths, n_assets))
    correlated = independent @ chol.T
    terminal = spots * np.exp((r - 0.5 * vols**2) * T + vols * np.sqrt(T) * correlated)
    basket = terminal @ weights
    payoff = np.maximum(basket - K, 0.0)
    discounted_payoff = np.exp(-r * T) * payoff
    price = discounted_payoff.mean()
    standard_error = discounted_payoff.std(ddof=1) / np.sqrt(n_paths)
    return price, standard_error


In [7]:
results = np.array([price_one(row, n_paths) for _, row in scenarios.iterrows()])
scenarios['mc_price'] = results[:, 0]
scenarios['mc_standard_error'] = results[:, 1]
scenarios[['mc_price', 'mc_standard_error']].describe()


,mc_price,mc_standard_error
count,2000.000000,2000.000000
mean,11.729461,0.369240
std,7.503935,0.179212
min,0.003843,0.002132
25%,5.565934,0.239504
50%,10.810328,0.355590
75%,16.782325,0.490219
max,35.561903,1.121171


In [8]:
output_dir = Path('artifacts')
scenarios.to_csv(output_dir / 'labeled_scenarios.csv', index=False)
print('saved Monte Carlo labels to', output_dir / 'labeled_scenarios.csv')


saved Monte Carlo labels to artifacts\labeled_scenarios.csv
